In [19]:
import pandas as pd
import numpy as np

# Load the Excel file
file_path = r'C:\Users\Xiaomi\Documents\GutHub\GEOMODELHACK24\Task2\Data\Данные_хакатон_actual.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display sheet names to understand the structure of the file
sheet_names = excel_data.sheet_names
sheet_names


['Общая информация', 'Характеристики', 'Матрица событий', 'Model']

In [2]:
# Load data from each sheet
general_info = excel_data.parse('Общая информация')
characteristics = excel_data.parse('Характеристики')
event_matrix = excel_data.parse('Матрица событий')
model_data = excel_data.parse('Model')

# Display the first few rows of each sheet to understand the data structure
general_info_head = general_info.head()
characteristics_head = characteristics.head()
event_matrix_head = event_matrix.head()
model_data_head = model_data.head()

#general_info_head, characteristics_head, event_matrix_head, model_data_head


0        2.673949
1        2.666466
2        2.659025
3        2.651626
4        2.644271
           ...   
65163    1.738539
65164    1.740328
65165    1.742211
65166    1.744189
65167    1.746262
Length: 65168, dtype: float64

In [23]:
# Split the data into training and testing sets
model_data['r'] = np.sqrt(model_data.X**2 + model_data.Y**2)
model_data['theta'] = np.arctan2(model_data.Y, model_data.X)
model_data['distance_0'] = np.sqrt((model_data.X - model_data.Y)**2)
model_data['x_angle'] = model_data.theta / (2 * np.pi)
model_data['slope'] = model_data.X / model_data.Y
X = model_data.drop(['Дельта', 'Питающий канал', 'Оползни', 'Конуса прорыва', 'Лопасти', 'Распределительный канал', 'Контуриты', 'is_well',	'well_name'], axis=1)

y = model_data[['Дельта', 'Питающий канал', 'Оползни', 'Конуса прорыва', 'Лопасти', 'Распределительный канал', 'Контуриты']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One-hot encode the target variables
y_train_encoded = pd.get_dummies(y_train)
y_test_encoded = pd.get_dummies(y_test)

# Train the neural network model
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(64, activation='relu'))
model.add(Dense(7, activation='softmax'))  # Output layer with 7 units for each column

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train_encoded, epochs=10, batch_size=32)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test_encoded)
print(f"Loss: {loss}, Accuracy: {accuracy}")

# Make predictions
predictions = model.predict(model_data[X.columns])

# Ensure the sum of probabilities for each location is equal to 1
predictions = predictions / predictions.sum(axis=1, keepdims=True)


Epoch 1/10
1630/1630 [==============================] - 3s 2ms/step - loss: 1.9640 - accuracy: 0.1457
Epoch 2/10
1630/1630 [==============================] - 3s 2ms/step - loss: 1.9529 - accuracy: 0.1432
Epoch 3/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9484 - accuracy: 0.1474
Epoch 4/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9469 - accuracy: 0.1492
Epoch 5/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9463 - accuracy: 0.1445
Epoch 6/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9460 - accuracy: 0.1471
Epoch 7/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9459 - accuracy: 0.1502
Epoch 8/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9459 - accuracy: 0.1458
Epoch 9/10
1630/1630 [==============================] - 2s 1ms/step - loss: 1.9459 - accuracy: 0.1472
Epoch 10/10
408/408 [==============================] - 0s 1ms/step - loss: 1.9459 

In [24]:
# Save the results
result_data = pd.concat([model_data[['X', 'Y']], pd.DataFrame(predictions, columns=['Дельта', 'Питающий канал', 'Оползни', 'Конуса прорыва', 'Лопасти', 'Распределительный канал', 'Контуриты'])], axis=1)


In [29]:
result_data['is_well'] = model_data['is_well']

In [33]:
result_data[result_data.is_well == True] = model_data[result_data.is_well == True]

In [36]:
result_data.drop('is_well', axis=1, inplace=True)

In [37]:
result_data.to_csv(r'C:\Users\Xiaomi\Documents\GutHub\GEOMODELHACK24\Task2\Data\predicted_probabilities.csv', index=False)